# 23_descendant_languages_v2: What Survives Time?

**Lab report:** Invariant Specification Demystifies Translation

Notebook 17 examined script change:

```text
same language
different scripts
```

Notebook 23 examines language evolution using the real upstream GLOSSOPETRAE API:

```js
const lang = Glossopetrae.quick(42);
const daughter = lang.evolve({ centuries: 8 });
const family = lang.deriveFamily({ daughters: 3 });
```

Core question:

> What survives centuries to millennia of accumulated linguistic change?

## 1. Setup

Clone or reuse the upstream GLOSSOPETRAE repository. This notebook uses Python to run a small Node/ESM probe script inside the upstream repo.

In [ ]:
from pathlib import Path
import os
import re
import json
import csv
import subprocess
from collections import Counter, defaultdict
from itertools import combinations

ROOT = Path.cwd()
UPSTREAM_URL = "https://github.com/elder-plinius/GLOSSOPETRAE.git"
UPSTREAM_DIR = ROOT / "GLOSSOPETRAE"

if not UPSTREAM_DIR.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
else:
    print("Upstream repo already present:", UPSTREAM_DIR)

FIGURES = ROOT / "figures"
DATA = ROOT / "data"
FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

print("Repository exists:", UPSTREAM_DIR.exists())

## 2. Confirm the upstream API

This cell searches source and docs for the real API methods used by this notebook:

- `Glossopetrae.quick`
- `.evolve`
- `.deriveFamily`

In [ ]:
api_terms = ["Glossopetrae.quick", ".evolve", "deriveFamily", "translationEngine", "translateToConlang"]
TEXT_SUFFIXES = {
    ".py", ".md", ".txt", ".yaml", ".yml", ".json", ".toml",
    ".js", ".mjs", ".ts", ".html", ".css", ".csv", ".tsv"
}

api_hits = []

for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    for term in api_terms:
        if term in text:
            api_hits.append({
                "path": str(path.relative_to(UPSTREAM_DIR)),
                "term": term,
                "count": text.count(term),
            })

try:
    import pandas as pd
    api_df = pd.DataFrame(api_hits).sort_values(["term", "path"])
    display(api_df)
    api_df.to_csv(DATA / "23_v2_api_hits.csv", index=False)
except Exception:
    for row in api_hits:
        print(row)

## 3. Run a real language-family probe

The probe creates:

1. a seeded proto-language,
2. a translation probe,
3. an evolved daughter language,
4. a three-daughter language family.

The script summarizes objects instead of dumping large audio, SVG, or deeply nested fields.

In [ ]:
runner = UPSTREAM_DIR / "spec_family_probe.mjs"

runner.write_text(r'''
import { Glossopetrae } from './src/Glossopetrae.js';

function summarizeObject(obj, depth = 0) {
  if (obj === null || obj === undefined) return obj;
  if (depth > 3) return '[depth-limit]';

  if (Array.isArray(obj)) {
    return obj.slice(0, 20).map(x => summarizeObject(x, depth + 1));
  }

  if (typeof obj === 'object') {
    const out = {};
    for (const key of Object.keys(obj).slice(0, 80)) {
      const value = obj[key];
      if (typeof value === 'function') continue;
      if (key.toLowerCase().includes('audio')) continue;
      if (key.toLowerCase().includes('wav')) continue;
      out[key] = summarizeObject(value, depth + 1);
    }
    return out;
  }

  if (typeof obj === 'string' && obj.length > 800) {
    return obj.slice(0, 800) + '...';
  }

  return obj;
}

function safeCall(label, fn) {
  try {
    return { ok: true, value: summarizeObject(fn()) };
  } catch (err) {
    return { ok: false, error: String(err), stack: String(err.stack || '') };
  }
}

const seed = 42;
const centuries = 8;
const daughters = 3;

const lang = Glossopetrae.quick(seed);

const translation = safeCall('translation', () =>
  lang.translationEngine.translateToConlang("The warrior sees the mountain.")
);

const daughter = safeCall('daughter', () =>
  lang.evolve({ centuries })
);

const family = safeCall('family', () =>
  lang.deriveFamily({ daughters })
);

const result = {
  seed,
  centuries,
  daughters,
  proto_language_keys: Object.keys(lang),
  proto_language_summary: summarizeObject(lang),
  translation,
  daughter,
  family,
};

console.log(JSON.stringify(result, null, 2));
''')

result = subprocess.run(
    ["node", str(runner.name)],
    cwd=UPSTREAM_DIR,
    text=True,
    capture_output=True,
)

print("returncode:", result.returncode)

if result.stderr:
    print("STDERR:")
    print(result.stderr[:5000])

print("STDOUT:")
print(result.stdout[:5000])

if result.returncode != 0:
    raise RuntimeError("Node runner failed. Check STDERR above.")

family_run = json.loads(result.stdout)

out_path = DATA / "23_v2_real_family_run_seed42.json"
out_path.write_text(json.dumps(family_run, indent=2))
print("Saved:", out_path)

## 4. Normalize the run

Convert the raw API output into the notebook variables used by the analysis.

In [ ]:
proto_language = {
    "seed": family_run.get("seed"),
    "keys": family_run.get("proto_language_keys", []),
    "summary": family_run.get("proto_language_summary", {}),
}

translation_probe = family_run.get("translation", {})
daughter_run = family_run.get("daughter", {})
family_generation = family_run.get("family", {})

evolution_rules = {
    "centuries": family_run.get("centuries"),
    "method": "lang.evolve({ centuries })",
}

correspondence = {
    "translation_probe_ok": translation_probe.get("ok"),
    "daughter_ok": daughter_run.get("ok"),
    "family_ok": family_generation.get("ok"),
    "method": "seeded proto-language with evolved descendants",
}

rows = [
    {
        "object": "proto_language",
        "method": "Glossopetrae.quick(seed)",
        "observed": bool(proto_language["keys"]),
        "summary": ", ".join(proto_language["keys"][:20]),
    },
    {
        "object": "translation_probe",
        "method": "lang.translationEngine.translateToConlang(sentence)",
        "observed": translation_probe.get("ok"),
        "summary": str(translation_probe.get("value") or translation_probe.get("error"))[:500],
    },
    {
        "object": "daughter_language",
        "method": "lang.evolve({ centuries: 8 })",
        "observed": daughter_run.get("ok"),
        "summary": str(daughter_run.get("value") or daughter_run.get("error"))[:500],
    },
    {
        "object": "language_family",
        "method": "lang.deriveFamily({ daughters: 3 })",
        "observed": family_generation.get("ok"),
        "summary": str(family_generation.get("value") or family_generation.get("error"))[:500],
    },
]

try:
    import pandas as pd
    run_df = pd.DataFrame(rows)
    display(run_df)
    run_df.to_csv(DATA / "23_v2_real_run_summary.csv", index=False)
except Exception:
    for row in rows:
        print(row)

## 5. Inspect descendant and family structures

This section looks at the keys and top-level structure returned by `evolve` and `deriveFamily`.

The goal is to identify whether the outputs preserve explicit lineage, correspondence, or specification metadata.

In [ ]:
def top_level_table(obj, label):
    if not isinstance(obj, dict):
        return [{"object": label, "key": "<not dict>", "type": type(obj).__name__, "preview": str(obj)[:300]}]

    rows = []
    for key, value in obj.items():
        rows.append({
            "object": label,
            "key": key,
            "type": type(value).__name__,
            "preview": str(value)[:300],
        })
    return rows

structure_rows = []
for label, wrapped in [
    ("translation", translation_probe),
    ("daughter", daughter_run),
    ("family", family_generation),
]:
    if isinstance(wrapped, dict) and wrapped.get("ok"):
        structure_rows.extend(top_level_table(wrapped.get("value"), label))
    else:
        structure_rows.append({
            "object": label,
            "key": "error",
            "type": "error",
            "preview": str(wrapped.get("error") if isinstance(wrapped, dict) else wrapped)[:500],
        })

try:
    import pandas as pd
    structure_df = pd.DataFrame(structure_rows)
    display(structure_df)
    structure_df.to_csv(DATA / "23_v2_output_structure.csv", index=False)
except Exception:
    for row in structure_rows:
        print(row)

## 6. Descendant-language invariant table

This table scores candidate invariants from the real run.

Some entries may remain unresolved if the upstream output does not expose the relevant field directly.

In [ ]:
def contains_key_like(obj, terms):
    if isinstance(obj, dict):
        for k, v in obj.items():
            kl = str(k).lower()
            if any(t in kl for t in terms):
                return True
            if contains_key_like(v, terms):
                return True
    elif isinstance(obj, list):
        return any(contains_key_like(x, terms) for x in obj)
    return False

raw_all = family_run

candidate_scores = [
    {
        "candidate": "lineage",
        "score": "observed" if contains_key_like(raw_all, ["lineage", "ancestor", "proto", "family", "daughter", "descendant"]) else "unresolved",
        "evidence": "family/daughter/descendant/proto metadata search",
    },
    {
        "candidate": "correspondence",
        "score": "observed" if contains_key_like(raw_all, ["correspond", "mapping", "translate", "translation"]) else "unresolved",
        "evidence": "translation/mapping/correspondence metadata search",
    },
    {
        "candidate": "meaning / concept",
        "score": "observed" if contains_key_like(raw_all, ["meaning", "concept", "semantic", "gloss"]) else "unresolved",
        "evidence": "meaning/concept/semantic/gloss metadata search",
    },
    {
        "candidate": "phonology / sound",
        "score": "observed" if contains_key_like(raw_all, ["phon", "sound", "syllable", "vowel", "consonant"]) else "unresolved",
        "evidence": "phonology/sound metadata search",
    },
    {
        "candidate": "lexicon / words",
        "score": "observed" if contains_key_like(raw_all, ["lex", "word", "vocab", "root"]) else "unresolved",
        "evidence": "lexicon/word metadata search",
    },
    {
        "candidate": "grammar",
        "score": "observed" if contains_key_like(raw_all, ["grammar", "syntax", "morph"]) else "unresolved",
        "evidence": "grammar/syntax/morphology metadata search",
    },
    {
        "candidate": "script / glyphs",
        "score": "observed" if contains_key_like(raw_all, ["script", "glyph", "orthograph", "writing"]) else "unresolved",
        "evidence": "script/glyph/orthography metadata search",
    },
]

try:
    import pandas as pd
    scores_df = pd.DataFrame(candidate_scores)
    display(scores_df)
    scores_df.to_csv(DATA / "23_v2_invariant_scores.csv", index=False)
except Exception:
    for row in candidate_scores:
        print(row)

## 7. Family tree figure

This figure summarizes the experiment.

The real run uses a seeded proto-language, applies evolution for eight centuries, and derives a small family of daughter languages.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
ax.axis("off")

nodes = {
    "Seeded Specification\\nseed = 42": (0.50, 0.92),
    "Proto Language\\nGlossopetrae.quick": (0.50, 0.74),
    "Evolution\\n8 centuries": (0.32, 0.54),
    "Family Derivation\\n3 daughters": (0.68, 0.54),
    "Daughter\\nlang.evolve": (0.25, 0.32),
    "Family A": (0.55, 0.32),
    "Family B": (0.70, 0.32),
    "Family C": (0.85, 0.32),
    "Invariant Candidates\\nlineage + correspondence": (0.50, 0.12),
}

for label, (x, y) in nodes.items():
    ax.text(
        x, y, label,
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.45", linewidth=1.2),
        fontsize=10
    )

edges = [
    ("Seeded Specification\\nseed = 42", "Proto Language\\nGlossopetrae.quick"),
    ("Proto Language\\nGlossopetrae.quick", "Evolution\\n8 centuries"),
    ("Proto Language\\nGlossopetrae.quick", "Family Derivation\\n3 daughters"),
    ("Evolution\\n8 centuries", "Daughter\\nlang.evolve"),
    ("Family Derivation\\n3 daughters", "Family A"),
    ("Family Derivation\\n3 daughters", "Family B"),
    ("Family Derivation\\n3 daughters", "Family C"),
    ("Daughter\\nlang.evolve", "Invariant Candidates\\nlineage + correspondence"),
    ("Family A", "Invariant Candidates\\nlineage + correspondence"),
    ("Family B", "Invariant Candidates\\nlineage + correspondence"),
    ("Family C", "Invariant Candidates\\nlineage + correspondence"),
]

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        "",
        xy=(x2, y2 + 0.04),
        xytext=(x1, y1 - 0.04),
        arrowprops=dict(arrowstyle="->", linewidth=1)
    )

path = FIGURES / "23_v2_family_tree.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
print("Saved:", path)
plt.show()

## 8. Pairwise-family similarity scaffold

If the upstream output exposes comparable lists such as words, concepts, phonemes, roots, or examples, this cell computes simple Jaccard similarities.

If fields are not exposed uniformly, the table will remain empty, which is itself useful: the next step would be to write a focused extractor for the upstream object shape.

In [ ]:
def collect_values_by_key(obj, key_terms):
    values = []

    def walk(x):
        if isinstance(x, dict):
            for k, v in x.items():
                kl = str(k).lower()
                if any(term in kl for term in key_terms):
                    if isinstance(v, (list, tuple, set)):
                        values.extend([str(item) for item in v])
                    elif isinstance(v, dict):
                        values.extend([str(item) for item in v.keys()])
                    else:
                        values.append(str(v))
                walk(v)
        elif isinstance(x, list):
            for item in x:
                walk(item)

    walk(obj)
    return values

family_value = family_generation.get("value") if isinstance(family_generation, dict) else None

# Try to extract child-like objects from common shapes.
family_members = {}

if isinstance(family_value, dict):
    for k, v in family_value.items():
        kl = str(k).lower()
        if any(term in kl for term in ["daughter", "child", "branch", "language", "family"]):
            if isinstance(v, list):
                for i, item in enumerate(v):
                    family_members[f"{k}_{i}"] = item
            elif isinstance(v, dict):
                for child_k, child_v in v.items():
                    family_members[f"{k}_{child_k}"] = child_v

if not family_members and isinstance(family_value, list):
    for i, item in enumerate(family_value):
        family_members[f"member_{i}"] = item

def jaccard(a, b):
    a = set(a)
    b = set(b)
    if not a and not b:
        return None
    return len(a & b) / len(a | b)

similarity_rows = []
fields = {
    "words": ["word", "lex", "vocab", "root"],
    "concepts": ["concept", "meaning", "semantic", "gloss"],
    "sounds": ["phon", "sound", "syllable", "vowel", "consonant"],
}

for left, right in combinations(family_members.keys(), 2):
    lobj = family_members[left]
    robj = family_members[right]
    for field, terms in fields.items():
        lvals = collect_values_by_key(lobj, terms)
        rvals = collect_values_by_key(robj, terms)
        similarity_rows.append({
            "left": left,
            "right": right,
            "field": field,
            "left_count": len(set(lvals)),
            "right_count": len(set(rvals)),
            "jaccard": jaccard(lvals, rvals),
        })

try:
    import pandas as pd
    sim_df = pd.DataFrame(similarity_rows)
    display(sim_df)
    sim_df.to_csv(DATA / "23_v2_family_similarity.csv", index=False)
except Exception:
    for row in similarity_rows:
        print(row)

print("Family members detected:", list(family_members.keys()))

## 9. Result statement

The exact empirical result depends on the upstream object fields exposed by the run.

At minimum, this notebook now verifies that GLOSSOPETRAE supports a seeded proto-language, evolution over a specified time interval, and family derivation using the programmatic API.

The core invariant candidates are:

- lineage,
- correspondence,
- meaning,
- specification constraints.

The strongest result would be explicit descendant metadata showing that different daughter languages remain linked by a shared ancestor or specification.

## 10. Hypothesis 4

**Hypothesis 4**

Language evolution changes observable forms.

Sounds change.

Words change.

Grammar and meaning may drift.

Scripts may change independently.

Yet descendant languages can retain correspondence relationships inherited from a common specification.

Lineage may be a stronger invariant than individual words, sounds, or glyphs.

## 11. Questions for Notebook 29

Notebook 29 should unify the three transformations:

```text
translation
script change
language evolution
```

Question:

> What invariant explains all three?

Candidate answers:

- specification,
- meaning,
- correspondence,
- lineage,
- constraint structure.

Notebook 29 should score these candidates across every transformation studied so far.